# Backprop by Hand, Checked Against a Reference Solution

**Topic:** Matrix calculus &nbsp;·&nbsp; **Difficulty:** Medium

Autograd is wonderful right up to the moment it disagrees with you and you have
no idea which of you is wrong. The cure is to have done it by hand at least once.

Two layers, `tanh` in the middle, softmax cross-entropy at the end. Derive every
gradient, then let the notebook check each one against reference values.

## The data (created in the next cell)

- `X` `(64, 12)`, `y` `(64,)` with 5 classes.
- `W1` `(12, 20)`, `b1` `(20,)`, `W2` `(20, 5)`, `b2` `(5,)`.
- Forward pass: `H = tanh(X @ W1 + b1)`, `Z = H @ W2 + b2`, mean softmax CE loss.

## Your task

With **NumPy only**:

1. Run the forward pass and compute the mean cross-entropy `loss`.
2. Derive and compute the gradients `gW1`, `gb1`, `gW2`, `gb2`.
   Remember `d(tanh)/dz = 1 - tanh(z)^2` and that softmax+CE gives
   `dZ = (probs - onehot) / n`.
3. Set `answers` to a dict with `"loss"`, `"gW1"`, `"gb1"`, `"gW2"`, `"gb2"`.

## Evaluation

The loss and every gradient are checked against reference values (each array's sum
and one fixed random projection) to a relative tolerance of 1e-6. Shapes must match
exactly and any `NaN` or `inf` fails.

In [ ]:
import numpy as np

rng = np.random.default_rng(71)
n, d, h, k = 64, 12, 20, 5
X = rng.normal(0, 1, (n, d))
y = rng.integers(0, k, n)
W1 = rng.normal(0, 0.5, (d, h))
b1 = rng.normal(0, 0.1, h)
W2 = rng.normal(0, 0.5, (h, k))
b2 = rng.normal(0, 0.1, k)
print(f"X {X.shape} -> W1 {W1.shape} -> W2 {W2.shape}, {k} classes")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("loss", [], 2.2424039224022687, -0.3754636611792821, 1e-06),
    ("gW1", [12, 20], -0.5885664813528535, -0.3700211227460935, 1e-06),
    ("gb1", [20], 0.06883462061102641, -0.25538640425564185, 1e-06),
    ("gW2", [20, 5], -1.1102230246251565e-16, -0.4672619791619861, 1e-06),
    ("gb2", [5], 0.0, 0.13261926731015045, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Work backwards. `dZ = (probs - onehot)/n` is the softmax+CE shortcut. Then
`gW2 = H.T @ dZ` and `gb2 = dZ.sum(0)`. Push through the `tanh`:
`dH = dZ @ W2.T`, `dA = dH * (1 - H**2)`, and finally `gW1 = X.T @ dA`,
`gb1 = dA.sum(0)`. Every shape is forced: if a product does not conform, the
derivation is wrong.

```python
import numpy as np

n = X.shape[0]
A = X @ W1 + b1
H = np.tanh(A)
Z = H @ W2 + b2

Zs = Z - Z.max(axis=1, keepdims=True)
expZ = np.exp(Zs)
probs = expZ / expZ.sum(axis=1, keepdims=True)
loss = float(-np.log(probs[np.arange(n), y]).mean())

onehot = np.zeros_like(probs)
onehot[np.arange(n), y] = 1.0
dZ = (probs - onehot) / n

gW2 = H.T @ dZ
gb2 = dZ.sum(axis=0)

dH = dZ @ W2.T
dA = dH * (1.0 - H ** 2)
gW1 = X.T @ dA
gb1 = dA.sum(axis=0)

answers = {"loss": loss, "gW1": gW1, "gb1": gb1, "gW2": gW2, "gb2": gb2}
```